In [1]:
import polars as pl
import tensorflow as tf
import numpy as np

NUM_OF_TOKENS = 12
NUM_OF_CNN_TOKENS = 4
BATCH_SIZE = 32


def prepare_data_lazy(file_path, num_prev=NUM_OF_TOKENS-1, cnn_tokens=NUM_OF_CNN_TOKENS, batch_size=BATCH_SIZE, shuffle=False):
    """
    Prepare data for the transformer model using Polars and lazy loading,
    with shuffling done by randomly selecting indices, and processing in batches.
    
    Args:
        file_path (str): Path to the CSV file.
        num_prev (int): Number of previous time frames to include.
        shuffle (bool): Whether to shuffle the data.
        batch_size (int): Number of rows to process in each batch.
    
    Yields:
        input_tensor (tf.Tensor): Tensor with shape (batch_size, num_prev + 1, 4).
        target_tensor (tf.Tensor): Tensor with shape (batch_size,).
    """
    # Load CSV lazily with Polars
    df_lazy = pl.scan_csv(file_path).select(['open_normalized', 'high_normalized', 'low_normalized', 'close_normalized', 'target'])

    # Collect the dataframe and determine total number of rows
    df_collected = df_lazy.collect()
    total_rows = df_collected.shape[0]

    # Create an array of indices to use for shuffling
    indices = list(range(num_prev, total_rows))
    # Shuffle indices if required
    if shuffle:
        np.random.shuffle(indices)

    input_list = []
    cnn_input_list = []
    target_list = []

    # Process the data in batches
    while indices:
        batch_indices = [indices.pop(0) for _ in range(min(batch_size, len(indices)))]

        for idx in batch_indices:
            # Fetch the previous `num_prev + 1` rows for the input based on the current index
            input_rows = df_collected[idx - num_prev:idx + 1, :-1].to_numpy()  # Exclude 'target' for input
            target_value = df_collected[idx, -1]  # Get 'target' for the target

            # Convert target to 0 if it's not 1
            target_value = 1 if target_value == 1 else 0

            input_list.append(input_rows)
            cnn_input_list.append(input_rows[:cnn_tokens])  # Select the first `cnn_tokens` rows for the CNN input
            target_list.append(target_value)

        # Convert lists to NumPy arrays
        input_array = np.array(input_list)
        cnn_input_array = np.array(cnn_input_list)
        cnn_input_array = np.expand_dims(cnn_input_array, axis=-1)
        target_array = np.array(target_list)

        # Convert NumPy arrays to TensorFlow tensors
        input_tensor = tf.convert_to_tensor(input_array, dtype=tf.float32)
        cnn_tensor = tf.convert_to_tensor(cnn_input_array, dtype=tf.float32)
        target_tensor = tf.convert_to_tensor(target_array, dtype=tf.int32)

        yield (input_tensor, cnn_tensor), target_tensor

        # Reset lists for the next batch
        input_list.clear()
        cnn_input_list.clear()
        target_list.clear()




# Rest of your model building and training code remains unchanged
def create_dataset_generator(file_path, batch_size=BATCH_SIZE, shuffle=False, repeat=False):
    dataset = tf.data.Dataset.from_generator(
        lambda: prepare_data_lazy(file_path, batch_size=batch_size, shuffle=shuffle),
        output_signature=(
            (tf.TensorSpec(shape=(None, NUM_OF_TOKENS, 4), dtype=tf.float32),
             tf.TensorSpec(shape=(None, NUM_OF_CNN_TOKENS, 4, 1), dtype=tf.float32)),
            tf.TensorSpec(shape=(None,), dtype=tf.int32)
        )
    )
    if repeat:
        dataset = dataset.repeat()
    return dataset


2024-10-07 13:14:03.845823: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-07 13:14:03.930923: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-07 13:14:05.343895: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
train_file = 'training/BTCUSD/train.csv'


train_dataset= create_dataset_generator(train_file, shuffle=True, repeat=True).prefetch(tf.data.AUTOTUNE)


2024-10-07 13:14:46.740897: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-07 13:14:46.808248: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-07 13:14:46.808309: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-07 13:14:46.810205: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-07 13:14:46.810257: I external/local_xla/xla/stream_executor

In [3]:
for (input_tensor, cnn_tensor), target_tensor in train_dataset.take(1):
    print("Input Tensor (Transformer input):")
    print(input_tensor.numpy())  # Convert to NumPy for easier readability

    print("\nCNN Tensor (CNN input):")
    print(cnn_tensor.numpy())  # Convert to NumPy for easier readability

    print("\nTarget Tensor (Labels):")
    print(target_tensor.numpy())  # Convert to NumPy for easier readability

Input Tensor (Transformer input):
[[[0.58754116 0.69894916 0.58754116 0.69894916]
  [0.6989937  0.768991   0.6989937  0.7123519 ]
  [0.7123519  0.7123519  0.7123519  0.7123519 ]
  ...
  [0.5788138  0.5788138  0.5788138  0.5788138 ]
  [0.64836586 0.6678244  0.64836586 0.66555345]
  [0.66559803 0.66777986 0.66559803 0.66773534]]

 [[0.8207591  0.8207591  0.7683689  0.77216023]
  [0.7527164  0.816523   0.7527164  0.80255437]
  [0.80255437 0.90998244 0.76275605 0.90350115]
  ...
  [0.7704552  0.8136107  0.7686019  0.79878426]
  [0.7993137  0.8964798  0.7688666  0.8964798 ]
  [0.91091436 0.92008555 0.85549533 0.86788595]]

 [[0.1736135  0.22666207 0.1736135  0.20144679]
  [0.20172235 0.2144678  0.16693076 0.19469514]
  [0.19469514 0.38346538 0.19455735 0.35935238]
  ...
  [0.3018946  0.3018946  0.20916294 0.20978299]
  [0.20978299 0.24485016 0.20978299 0.23224251]
  [0.2321736  0.2321736  0.12731656 0.18339649]]

 ...

 [[0.8560732  0.8627288  0.8377704  0.8610649 ]
  [0.85232943 0.86023295

2024-10-07 13:19:47.587287: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
